In [3]:
import os
from pathlib import Path
import pandas as pd
import requests

# Look for .env in the parent directory if your notebook is in a subfolder (e.g., notebooks/)
env_path = Path("../.env") if Path("../.env").exists() else Path(".env")
for line in env_path.read_text().splitlines() if env_path.exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())

API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
API_KEY = os.environ.get("RAWTREE_API_KEY")
TEAM = os.environ.get("TEAM_ID")

if not API_KEY:
    raise ValueError("RAWTREE_API_KEY is missing! Check your .env file location.")

http = requests.Session()
http.headers["Authorization"] = f"Bearer {API_KEY}"

In [4]:
def query(sql: str) -> pd.DataFrame:
    """Helper function to execute SQL queries against the RawTree API and return a DataFrame."""
    r = http.post(f"{API}/v1/query", json={"sql": sql}, timeout=600)
    r.raise_for_status()
    data = r.json().get("data", [])
    return pd.DataFrame(data)

print("Fetching all prediction data sources from RawTree...")

Fetching all prediction data sources from RawTree...


In [5]:
# All-time best sellers ranking (derived from historical purchases)
sales_sql = """
SELECT product_id, count() AS n
FROM shop.history_events
WHERE event_type = 'purchase'
GROUP BY product_id
ORDER BY n DESC
"""
best_df = query(sales_sql)
BEST_SELLERS: list[str] = best_df["product_id"].tolist() if not best_df.empty else []

# Current out-of-stock items (recommending these makes an answer invalid)
stock_sql = """
SELECT product_id::String AS product_id
FROM shop.catalog_updates
WHERE change::String IN ('out_of_stock', 'back_in_stock')
GROUP BY product_id
HAVING argMax(change::String, ts::DateTime64(3)) = 'out_of_stock'
"""
stock_df = query(stock_sql)
OUT_OF_STOCK: set[str] = set(stock_df["product_id"]) if not stock_df.empty else set()



In [6]:
# Co-views: For every product, the top 20 products most viewed in the same sessions
coviews_sql = """
SELECT a.product_id AS p, b.product_id AS q, count() AS n
FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view') AS a
INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view') AS b
  ON a.session_id = b.session_id
WHERE a.product_id != b.product_id
GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p
"""
coviews_df = query(coviews_sql)
COVIEWS_DICT: dict[str, list[str]] = (
    coviews_df.groupby("p").q.apply(list).to_dict() if not coviews_df.empty else {}
)


In [7]:

# Active incoming recommendation requests queue
requests_sql = """
SELECT request_id, ts, user_id, context, recent_views
FROM shop.requests
ORDER BY ts DESC
LIMIT 100
"""
RECENT_REQUESTS: pd.DataFrame = query(requests_sql)



In [8]:

# Historical session logs (contains impressions, clicks, propensity scores)[cite: 1]
history_sessions_sql = """
SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
FROM shop.history_sessions
ORDER BY ts DESC
LIMIT 10000
"""
HISTORY_SESSIONS: pd.DataFrame = query(history_sessions_sql)

# Granular historical event logs (individual views and purchases for matrix factorization / item similarity)
history_events_sql = """
SELECT session_id, user_id, event_type, product_id, ts
FROM shop.history_events
ORDER BY ts DESC
LIMIT 10000
"""
HISTORY_EVENTS: pd.DataFrame = query(history_events_sql)


In [9]:
catalog = query("SELECT * FROM shop.catalog")
feedback_df = query(f"SELECT * FROM {TEAM}.feedback ORDER BY ts DESC LIMIT 1000")

In [10]:
# Summary of loaded variables
if __name__ == "__main__":
    print("\n--- Data Loaded Successfully ---")
    print(f"• BEST_SELLERS: {len(BEST_SELLERS)} products loaded")
    print(f"• OUT_OF_STOCK: {len(OUT_OF_STOCK)} items currently unavailable")
    print(f"• COVIEWS_DICT: {len(COVIEWS_DICT)} products have co-view maps")
    print(f"• RECENT_REQUESTS: {len(RECENT_REQUESTS)} live request rows fetched")
    print(f"• HISTORY_SESSIONS: {len(HISTORY_SESSIONS)} session rows fetched")
    print(f"• HISTORY_EVENTS: {len(HISTORY_EVENTS)} event rows fetched")


--- Data Loaded Successfully ---
• BEST_SELLERS: 4809 products loaded
• OUT_OF_STOCK: 36 items currently unavailable
• COVIEWS_DICT: 5000 products have co-view maps
• RECENT_REQUESTS: 100 live request rows fetched
• HISTORY_SESSIONS: 10000 session rows fetched
• HISTORY_EVENTS: 10000 event rows fetched


---

In [11]:
bestSellers = pd.DataFrame(BEST_SELLERS)
outOfStock = pd.DataFrame(OUT_OF_STOCK)
coviewsDict = COVIEWS_DICT
recentRequests = pd.DataFrame(RECENT_REQUESTS)
historySessions = pd.DataFrame(HISTORY_SESSIONS)
historyEvents = pd.DataFrame(HISTORY_EVENTS)
catalog = catalog
feedback = feedback_df


In [16]:
feedback = query(f"SELECT * FROM {TEAM}.feedback ORDER BY ts DESC LIMIT 10")
feedback.head()

,clicked,fallback_reason,latency_ms,model_version,purchased,request_id,revenue_eur,served_by,shown,ts
0,,missing,None,,,r_45ae13b2b2ae6692,0,fallback,"[p_adc3e6, p_69d40e, p_09a938, p_1036d3, p_e24...",2026-11-14 21:22:50.283000000
1,,missing,None,,,r_2dd3757233fb996c,0,fallback,"[p_adc3e6, p_69d40e, p_09a938, p_1036d3, p_e24...",2026-11-14 21:22:49.913000000
2,,missing,None,,,r_6815a7d70de37d76,0,fallback,"[p_adc3e6, p_69d40e, p_09a938, p_1036d3, p_e24...",2026-11-14 21:22:49.649000000
3,,missing,None,,,r_061d392101c3e554,0,fallback,"[p_adc3e6, p_69d40e, p_09a938, p_1036d3, p_e24...",2026-11-14 21:22:49.250000000
4,,missing,None,,,r_f59a515f6f9e406d,0,fallback,"[p_adc3e6, p_69d40e, p_09a938, p_1036d3, p_e24...",2026-11-14 21:22:48.928000000


In [11]:
from featureConstructor import FeatureConstructor

In [12]:
import importlib
import featureConstructor
importlib.reload(featureConstructor)
from featureConstructor import FeatureConstructor

fc = FeatureConstructor(
    query_function=query,
    catalog=catalog,
    coview_dict=coviewsDict,
    best_sellers=bestSellers,
    out_of_stock=outOfStock,
)

# Optionally limit to 2,000 sessions/examples if generate_xy doesn't accept max_examples
sampled_interactions = (
    positive_interactions_df.sample(n=min(2000, len(positive_interactions_df)), random_state=42)
    # Or use .head(2000) to take the first 2,000 in order
)

X, y_click, y_buy, sample_weights = fc.generate_xy(
    interactions_df=sampled_interactions,
    history_sessions=historySessions,
)

AttributeError: module 'userFeatures' has no attribute 'prefetch'

In [ ]:
output_df = X.assign(
    y_click=y_click,
    y_buy=y_buy,
    sample_weights=sample_weights,
)
output_df.to_csv("xy_dataset.csv", index=False)

In [ ]:
print("Clicks:", y_click.sum(), f"({y_click.mean():.1%})")
print("Buys:  ", y_buy.sum(), f"({y_buy.mean():.1%})")

Clicks: 368 (10.0%)
Buys:   150 (4.1%)


In [ ]:
df = pd.read_csv("xy_dataset.csv")

df.shape

(3680, 97)

In [ ]:
from coviewFeatures import extract_coview_features_from_events

[]
{'coview_score': 0.0, 'coview_frequency': 0, 'coview_best_rank': 999, 'coview_triggered_by_last_view': 0, 'coview_avg_rank': 999.0, 'time_since_last_trigger_sec': 999999.0, 'has_coview_signal': 0}


In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Read the CSV and reconstruct X, targets, and weights
df = pd.read_csv("dataset.csv")

target_cols = ["y_click", "y_buy", "sample_weights"]
y_click = df["y_click"].to_numpy()
y_buy = df["y_buy"].to_numpy()
sample_weights = df["sample_weights"].to_numpy()
X = df.drop(columns=target_cols)

# CSV loses pandas 'category' dtypes; convert string/object columns back for LightGBM
for col in X.select_dtypes(include=["object", "string"]).columns:
    X[col] = X[col].astype("category")

# 2. Apply extra weight to interactions that resulted in a purchase
BUY_WEIGHT_MULTIPLIER = 5.0
combined_weights = sample_weights * np.where(y_buy == 1, BUY_WEIGHT_MULTIPLIER, 1.0)
y_target = np.maximum(y_click, y_buy)

# 3. Train/validation split and model training
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X,
    y_target,
    combined_weights,
    test_size=0.2,
    random_state=42,
)

model = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    learning_rate=0.05,
    num_leaves=31,
    n_estimators=1000,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train,
    sample_weight=w_train,
    eval_set=[(X_val, y_val)],
    eval_sample_weight=[w_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

scores = model.predict_proba(X_val)[:, 1]

/home/camoraz/Code/team18/team-18/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1233, number of negative: 22767
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015370 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21089
[LightGBM] [Info] Number of data points in the train set: 24000, number of used features: 96
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.007078 -> initscore=-4.943690
[LightGBM] [Info] Start training from score -4.943690


In [ ]:
model.booster_.save_model("lgbm_click_buy_model.txt")

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

# 1. Overall validation metrics (runs immediately with your existing y_val, w_val, scores)
print("=== Overall Validation Metrics ===")
print(f"Weighted ROC-AUC:   {roc_auc_score(y_val, scores, sample_weight=w_val):.4f}")
print(f"Unweighted ROC-AUC: {roc_auc_score(y_val, scores):.4f}")
print(f"Weighted PR-AUC:    {average_precision_score(y_val, scores, sample_weight=w_val):.4f}")
print(f"Unweighted PR-AUC:  {average_precision_score(y_val, scores):.4f}")

=== Overall Validation Metrics ===
Weighted ROC-AUC:   0.9880
Unweighted ROC-AUC: 0.8550
Weighted PR-AUC:    0.2423
Unweighted PR-AUC:  0.2104
